# Call MAS Service through Python

In [ ]:
import requests
import base64
import json
import pandas as pd
import sys
import os
import re

### Set Parameters

In [ ]:
# Logon Credentials. 
server= '127.0.0.1'
uid= 'sasdemo'
pwd= 'Orion123'
clientId= 'mas.client'
clientSecret= 'Orion123'

######################################
decision_name= 'getPatientData'
decision_id= '37cb79d4-17f6-4942-96aa-f82a4e70f7c9'
revision_id= 'f20d1f64-fb77-4df0-9d27-9f06b2934f81'

sasLibs= 'WORK PGDATA'

input_table= 'work.patient'
output_table= 'work.patient_out'


### Get Viya User Authorization Token

In [ ]:
loginCredentials={"grant_type":"password","username":uid,"password":pwd}
appBinding= clientId + ':' + clientSecret
appBinding64= base64.b64encode(bytes(appBinding, 'utf-8'))
url= "http://%s/SASLogon/oauth/token" % (server)

headers = {"Content-Type":"application/x-www-form-urlencoded", "Authorization":"Basic "}
headers["Authorization"]= "Basic " + appBinding64.decode('ascii') #appBinding64 is type bytes but need to convert to type str
response = requests.post(url, headers=headers, data=loginCredentials)

if response.status_code < 200 or response.status_code >= 300:
    print('Error receiving user token!')
    print(pd.DataFrame([response.json()]))
    sys.exit()
else:
    token= response.json()['access_token']

### Call ID Service

In [ ]:
url= "http://%s/decisions/flows/%s/revisions/%s/code?codeTarget=MICROANALYTICSERVICE&lookupMode=inline&isGeneratingRuleFiredColumn=false&rootPackageName=%s" % (server, decision_id, revision_id, decision_name)

headers = {"Content-Type":"application/json", "Authorization":"Bearer "}
headers["Authorization"]= "Bearer " + token
response = requests.get(url, headers=headers)

if response.status_code < 200 or response.status_code >= 300:
    print('Error calling MAS service!')
    print(response.json()['message'])
    pd.options.display.max_colwidth = -1
    print(pd.DataFrame(data= response.json()['details']))
    sys.exit()

In [ ]:
#save code to temp file
tmpFile= '/tmp/' + revision_id +'.tmp'
fTmp= open(tmpFile, 'w')
fTmp.write(response.text)
fTmp.close()

In [ ]:
#adjust code and move to ds2 file
codeFile= '/tmp/' + revision_id +'.ds2'
fCode= open(codeFile, 'w')

lastPackage= None
fTmp= open(tmpFile, 'r')

fCode.write('proc ds2 libs=(WORK ' +sasLibs +');\n')
for line in fTmp:
    #identify last package as we need the root package later
    if re.search("^\s*package\s+.+\s*\/\s*inline;", line):
        lastPackage= line
    
    #remove code to call record contact node and change statement to output record to create it later
    if line.find('st.recordContactHistory') >= 0:
        fCode.write(line[line.find('(')+1:line.find(',')] +'= ' +line[line.find(',')+1:line.find(')')] +';\n')
    elif line.strip() != 'dcl package masstate st();':
        fCode.write(line)
        
fTmp.close()
fCode.close()    

In [ ]:
#look where the root package is and modify as needed
fTmp= open(tmpFile, 'r')

bRootPackage= False
bExecuteMethod= False
executeMethod= ''
for line in fTmp:
    #find root package
    if line == lastPackage:
        bRootPackage= True
    #find execute function
    if bRootPackage and re.search("^\s*method\s+execute\s*\(", line):
        bExecuteMethod= True
    if bExecuteMethod:
        executeMethod+= line + ' '
    if bExecuteMethod and re.search("^\s*.*\s*\)\s*;", line):
        bExecuteMethod= False

fTmp.close()
os.remove(tmpFile)

#get parameters from root execute() method
executeMethodParameters= re.sub("^\s*method\s+execute\s*\(\s*", '', executeMethod)
executeMethodParameters= re.sub("\s*\)\s*;", '', executeMethodParameters)
executeMethodParameterList= executeMethodParameters.split(',')

print(executeMethodParameterList)

In [ ]:
parameterList= []
for parameter in executeMethodParameterList:
    parameterList.append(parameter.strip())
print(parameterList)

In [ ]:
#write modified ds2 file
dgList= []
outputParaDcl= ''
outputParaKeep= 'keep '
buffer= ''
i= 0
for parameter in parameterList:
    if parameter[len(parameter)-2] == '_':
        if parameter.find('package') > 0:
            parameterList[i]= parameter[0:len(parameter)-2] + '_dg"'
        else:
            parameterList[i]= parameter[0:len(parameter)-2] + '"'
        parameterList[i]= re.sub("^\s*.+\s+", '', parameterList[i]) 
    else:
        parameterList[i]= parameterList[i].replace('in_out ', '')
        
        buffer= parameterList[i].replace('varchar', 'varchar(1000)')
        
        if parameter.find('package') > 0:
            outputParaDcl+= 'dcl ' +buffer[0:len(buffer)-1] +'_dg"();\n    '
            buffer= buffer.replace('package', '')
            buffer= buffer.replace('datagrid', 'varchar(10485760)')
            outputParaDcl+= 'dcl ' +buffer +';\n    '
        else:
            outputParaDcl+= 'dcl ' +buffer +';\n    '
        
        packageVar= False
        if parameter.find('package') > 0:
            packageVar= True
            
        parameterList[i]= re.sub("^\s*.+\s+", '', parameterList[i]) 
        outputParaKeep+= parameterList[i] +' '
        
        if packageVar: 
            parameterList[i]= parameterList[i][0:len(parameterList[i])-1] + '_dg"'
            dgList.append(parameterList[i])        
        
    i+= 1
print(parameterList)
print(outputParaDcl)
print(outputParaKeep)

outputParaKeep+= ';'  

i= 0
callExecute= 'dcf.execute('
for parameter in parameterList:
    if i == 0:
        callExecute+= parameter
    else:
        callExecute+= ', ' +parameter
    i+= 1
callExecute+= ');'


In [ ]:
fCode= open(codeFile, 'a')
fCode.write('data ' +output_table +';\n')
fCode.write('    ' +outputParaKeep +'\n')
fCode.write('    dcl package ' +decision_name +' dcf();\n')
fCode.write('    ' +outputParaDcl +'\n')
fCode.write('    method run();\n')
fCode.write('        set ' +input_table +';\n')
fCode.write('        ' +callExecute +'\n')
for dg in dgList:
    fCode.write('        ' +dg[0:len(dg)-4] +'"= ' +dg +'.serialize();\n')
for dg in dgList:
    fCode.write('        ' +dg +'.clear();\n')
fCode.write('        end;\n    run;\nquit;\n')
fCode.close()